# Inez CO₂ Storage Capacity Assessment

This notebook runs a probabilistic static storage-capacity assessment for the **Haldager Sand, Gassum and Skagerrak** reservoirs. It reports each reservoir separately and adds their simulated capacities trial by trial to obtain the combined Inez distribution.

$$SC = GRV \times (N/G) \times \phi \times \rho_{CO_2} \times S_{eff}$$

Change the values in the **Editable inputs** cell, then choose **Runtime → Run all**. The code used for calculations and figures is collapsed by default; its tables, plots and results remain visible.


In [ ]:
#@title Install dependencies { display-mode: "form" }
"""Install the latest package and plotting tools from GitHub."""
%pip install -q --upgrade --force-reinstall --no-cache-dir --no-deps "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git"
%pip install -q matplotlib pandas


In [ ]:
#@title Load analysis tools { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import Distribution, SimulationResult, StorageSite, simulate

plt.style.use("seaborn-v0_8-whitegrid")


## Editable inputs

Enter minimum, most likely and maximum values. Fractions are decimals: `0.07` means 7%. The values come from GEUS Report 2022/29, Tables 8.1.5.1–8.1.5.3.

**Source note:** the Inez Gassum reservoir-specific table gives a 7% storage-efficiency mode. This value reproduces the detailed published result more closely than the 10% mentioned in the report's general text.


In [ ]:
#@title Editable inputs — keep this cell visible
inputs = {
    "Haldager Sand": {
        "GRV (km³)": (0.114, 0.417, 1.676), "N/G": (0.200, 0.317, 0.500),
        "Porosity": (0.200, 0.255, 0.300), "CO₂ density (kg/m³)": (609.6, 641.7, 705.9),
        "Storage efficiency": (0.050, 0.100, 0.150)},
    "Gassum Formation": {
        "GRV (km³)": (11.059, 28.076, 49.666), "N/G": (0.4696, 0.5870, 0.7044),
        "Porosity": (0.1624, 0.2030, 0.2436), "CO₂ density (kg/m³)": (611.8, 644.0, 708.4),
        "Storage efficiency": (0.050, 0.070, 0.150)},
    "Skagerrak Formation": {
        "GRV (km³)": (3.7105, 8.781, 12.915), "N/G": (0.3056, 0.3820, 0.4584),
        "Porosity": (0.1624, 0.2030, 0.2436), "CO₂ density (kg/m³)": (607.2, 639.2, 703.1),
        "Storage efficiency": (0.050, 0.100, 0.150)},
}


## Input parameter table

This table is generated from the editable values above, so it updates automatically when an input changes.


In [ ]:
#@title Show input parameter table { display-mode: "form" }
rows = [[reservoir, parameter, "PERT", *values] for reservoir, parameters in inputs.items() for parameter, values in parameters.items()]
input_table = pd.DataFrame(rows, columns=["Reservoir", "Parameter", "Distribution", "Minimum", "Mode", "Maximum"])
input_table


In [ ]:
#@title Run the Monte Carlo simulation { display-mode: "form" }
def make_site(name, values):
    return StorageSite(
        name=f"Inez – {name}",
        grv=Distribution.pert(*values["GRV (km³)"]),
        net_to_gross=Distribution.pert(*values["N/G"]),
        porosity=Distribution.pert(*values["Porosity"]),
        co2_density=Distribution.pert(*values["CO₂ density (kg/m³)"]),
        storage_efficiency=Distribution.pert(*values["Storage efficiency"]),
    )

iterations = 100_000
results = {name: simulate(make_site(name, values), iterations, seed=42+i) for i, (name, values) in enumerate(inputs.items())}
combined_capacity = np.sum([result.capacity_mt for result in results.values()], axis=0)
combined = SimulationResult("Inez – combined reservoirs", combined_capacity, {})


## Results and comparison with GEUS

P90/P50/P10 are shown separately for every reservoir and for combined Inez. For the combined result, the three reservoir capacities are added in every trial and percentiles are calculated afterwards; reservoir percentiles are not added together.


In [ ]:
#@title Show capacity results { display-mode: "form" }
published = {
    "Haldager Sand": (1.2, 2.8, 5.5, 3.1),
    "Gassum Formation": (103.8, 168.1, 263.7, 177.6),
    "Skagerrak Formation": (27.4, 42.1, 60.7, 43.2),
    "Combined Inez": (148.6, 216.2, 310.2, 224.8),
}
all_results = {**results, "Combined Inez": combined}
comparison_rows = []
for name, result in all_results.items():
    s = result.summary()
    simulated = (s["p90_mt"], s["p50_mt"], s["p10_mt"], s["mean_mt"])
    comparison_rows.append([name, *simulated, *published[name]])
comparison_table = pd.DataFrame(comparison_rows, columns=["Reservoir", "P90 simulated", "P50 simulated", "P10 simulated", "Mean simulated", "P90 GEUS", "P50 GEUS", "P10 GEUS", "Mean GEUS"])
comparison_table.round(2)


The notebook and GEUS values should be close but not necessarily identical. Both use the same static volumetric equation and independent PERT inputs. Small differences are expected because the report does not state its random seed, iteration count or exact PERT implementation.


## Input uncertainty distributions

Every parameter histogram identifies the exact **minimum**, **mode** and **maximum** entered in the GEUS input table, plus the **mean** of the Monte Carlo samples.


In [ ]:
#@title Show input uncertainty distributions { display-mode: "form" }
labels = {
    "grv_km3": "GRV (km³)",
    "net_to_gross": "Net-to-gross",
    "porosity": "Porosity",
    "co2_density_kg_m3": "CO₂ density (kg/m³)",
    "storage_efficiency": "Storage efficiency",
}
input_lookup = {
    "grv_km3": "GRV (km³)",
    "net_to_gross": "N/G",
    "porosity": "Porosity",
    "co2_density_kg_m3": "CO₂ density (kg/m³)",
    "storage_efficiency": "Storage efficiency",
}
line_styles = [("Minimum", "#1f77b4", ":"), ("Mode", "#2ca02c", "--"), ("Mean", "#ff7f0e", "-"), ("Maximum", "#d62728", ":")]
fig, axes = plt.subplots(5, 3, figsize=(16, 18))
for column, (reservoir_name, result) in enumerate(results.items()):
    for row, (name, values) in enumerate(result.inputs.items()):
        ax = axes[row, column]
        minimum, mode, maximum = inputs[reservoir_name][input_lookup[name]]
        mean = float(np.mean(values))
        reference_values = [minimum, mode, mean, maximum]
        ax.hist(values, bins=45, color="#b9d7f0", edgecolor="white", alpha=0.9)
        for (line_name, color, linestyle), reference in zip(line_styles, reference_values):
            ax.axvline(reference, color=color, linestyle=linestyle, linewidth=1.6, label=f"{line_name}: {reference:.4g}")
        ax.set_title(f"{reservoir_name}\n{labels[name]}")
        ax.set_ylabel("Simulations")
        ax.legend(fontsize=7, frameon=True, loc="upper right")
fig.suptitle("Inez – input uncertainty distributions and values used", fontsize=16, y=1.01)
fig.tight_layout()
plt.show()


## Storage-capacity distributions: separate and combined

The four panels use the same format: grey bars are simulated capacities, the orange line is a fitted lognormal probability-density curve, and the red line is cumulative exceedance probability. P90 is the conservative estimate, P50 the median and P10 the upside estimate.


In [ ]:
#@title Show separate and combined histograms with cumulative curves { display-mode: "form" }
def plot_capacity_distribution(ax_density, result_name, result):
    values = np.asarray(result.capacity_mt, dtype=float)
    summary = result.summary()
    ax_density.hist(values, bins=50, density=True, color="#d9d9d9", edgecolor="white", linewidth=0.5, label="Simulated capacity")
    log_values = np.log(values)
    mu, sigma = float(np.mean(log_values)), float(np.std(log_values, ddof=1))
    x = np.linspace(float(np.min(values)), float(np.max(values)), 500)
    fitted_density = np.exp(-0.5 * ((np.log(x) - mu) / sigma) ** 2) / (x * sigma * np.sqrt(2 * np.pi))
    ax_density.plot(x, fitted_density, color="#f39c12", linewidth=2.0, label="Fitted density")
    ax_cumulative = ax_density.twinx()
    capacity_sorted = np.sort(values)
    exceedance_pct = (1 - np.arange(1, capacity_sorted.size + 1) / (capacity_sorted.size + 1)) * 100
    ax_cumulative.plot(capacity_sorted, exceedance_pct, color="#e31a1c", linewidth=1.8, label="Cumulative exceedance")
    markers = [("P90", "p90_mt", 90, "#b2182b"), ("P50", "p50_mt", 50, "#ff8c42"), ("P10", "p10_mt", 10, "#b2182b")]
    for marker_label, key, probability, color in markers:
        marker_value = summary[key]
        ax_density.axvline(marker_value, color=color, linestyle="--", linewidth=1.1, alpha=0.9)
        ax_cumulative.plot(marker_value, probability, "o", color=color, markersize=4)
        vertical_offset = 8 if marker_label != "P10" else -14
        ax_cumulative.annotate(f"{marker_label}: {marker_value:.1f} Mt", (marker_value, probability), xytext=(5, vertical_offset), textcoords="offset points", color=color, fontsize=8)
    ax_density.text(0.98, 0.96, f"Mean: {summary['mean_mt']:.1f} Mt\nSD: {np.std(values, ddof=1):.1f} Mt", transform=ax_density.transAxes, ha="right", va="top", color="#b56500", fontsize=8)
    ax_density.set_title(result_name)
    ax_density.set_xlabel("Storage capacity (Mt CO₂)")
    ax_density.set_ylabel("Probability density")
    ax_cumulative.set_ylabel("Exceedance probability (%)", color="#e31a1c")
    ax_cumulative.set_ylim(0, 100)
    ax_cumulative.tick_params(axis="y", colors="#e31a1c")
    handles_1, legend_1 = ax_density.get_legend_handles_labels()
    handles_2, legend_2 = ax_cumulative.get_legend_handles_labels()
    ax_density.legend(handles_1 + handles_2, legend_1 + legend_2, loc="upper center", fontsize=7)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, (result_name, result) in zip(axes.flat, all_results.items()):
    plot_capacity_distribution(ax, result_name, result)
fig.suptitle("Inez storage-capacity distributions – separate reservoirs and combined site", fontsize=16, y=1.01)
fig.tight_layout()
plt.show()


## Combined exceedance curve

This larger standalone view summarizes combined Inez. P90 is the capacity that has a 90% probability of being exceeded; P10 is the upside estimate.


In [ ]:
#@title Show combined exceedance curve { display-mode: "form" }
fig, ax = combined.plot_exceedance()
plt.show()


## Combined linear capacity confidence ranges

The colored bar summarizes conservative, central and upside capacity ranges for combined Inez. These are probabilistic static capacity estimates, not booked reserves.


In [ ]:
#@title Show combined linear capacity confidence ranges { display-mode: "form" }
fig, ax = combined.plot_capacity_ranges()
plt.show()


## One-at-a-time tornado charts: three separate and one combined

For each bar, one input moves from its P90 input value (10th sample percentile) to its P10 input value (90th sample percentile), while all other inputs stay at their sampled means. The labels on the bar are the resulting low and high capacities—not correlation coefficients.

The **Gassum Formation** panel is the direct comparison with the GEUS tornado (Figure 8.2.2). GEUS does not publish equivalent tornado figures for Haldager Sand, Skagerrak or combined Inez; those three panels are additional analyses made by this notebook using the same method.


In [ ]:
#@title Show separate and combined one-at-a-time tornado charts { display-mode: "form" }
def capacity_from_inputs(parameter_values):
    return float(np.prod(list(parameter_values.values())))

def reservoir_tornado(result):
    sample_means = {name: float(np.mean(samples)) for name, samples in result.inputs.items()}
    baseline = capacity_from_inputs(sample_means)
    rows = []
    for name, samples in result.inputs.items():
        low_inputs, high_inputs = sample_means.copy(), sample_means.copy()
        low_inputs[name] = float(np.quantile(samples, 0.10))
        high_inputs[name] = float(np.quantile(samples, 0.90))
        rows.append((labels[name], capacity_from_inputs(low_inputs), capacity_from_inputs(high_inputs)))
    return baseline, rows

def draw_tornado(ax, title, baseline, rows):
    ordered_rows = sorted(rows, key=lambda row: row[2] - row[1])
    names = [row[0] for row in ordered_rows]
    lows = np.asarray([row[1] for row in ordered_rows])
    highs = np.asarray([row[2] for row in ordered_rows])
    widths = highs - lows
    bars = ax.barh(names, widths, left=lows, color="#5b9bd5", edgecolor="#2f5597", alpha=0.9)
    ax.axvline(baseline, color="#c00000", linestyle="--", linewidth=1.4, label=f"Base: {baseline:.1f} Mt")
    chart_range = max(float(np.max(highs) - np.min(lows)), 1e-12)
    padding = 0.10 * chart_range
    ax.set_xlim(float(np.min(lows) - padding), float(np.max(highs) + padding))
    for bar, low, high, width in zip(bars, lows, highs, widths):
        y = bar.get_y() + bar.get_height() / 2
        if width >= 0.16 * chart_range:
            ax.text(low + 0.03 * width, y, f"{low:.1f}", ha="left", va="center", color="white", fontsize=8, fontweight="bold")
            ax.text(high - 0.03 * width, y, f"{high:.1f}", ha="right", va="center", color="white", fontsize=8, fontweight="bold")
        else:
            ax.annotate(f"{low:.1f}", (low, y), xytext=(-4, 0), textcoords="offset points", ha="right", va="center", color="#1f1f1f", fontsize=7, fontweight="bold")
            ax.annotate(f"{high:.1f}", (high, y), xytext=(4, 0), textcoords="offset points", ha="left", va="center", color="#1f1f1f", fontsize=7, fontweight="bold")
    ax.set_title(title)
    ax.set_xlabel("Storage capacity (Mt CO₂)")
    ax.legend(fontsize=8, loc="lower right")

individual_tornado_data = {name: reservoir_tornado(result) for name, result in results.items()}
individual_baselines = {name: data[0] for name, data in individual_tornado_data.items()}
combined_baseline = sum(individual_baselines.values())
combined_rows = []
for reservoir_name, (reservoir_baseline, rows) in individual_tornado_data.items():
    other_reservoirs = combined_baseline - reservoir_baseline
    combined_rows.extend((f"{reservoir_name} — {name}", low + other_reservoirs, high + other_reservoirs) for name, low, high in rows)

fig, axes = plt.subplots(2, 2, figsize=(17, 12))
for ax, (reservoir_name, (baseline, rows)) in zip(axes.flat[:3], individual_tornado_data.items()):
    comparison_label = " (GEUS comparison)" if reservoir_name == "Gassum Formation" else ""
    draw_tornado(ax, f"{reservoir_name}{comparison_label}", baseline, rows)
draw_tornado(axes.flat[3], "Combined Inez – all three formations", combined_baseline, combined_rows)
fig.suptitle("Inez one-at-a-time capacity sensitivity", fontsize=16, y=1.01)
fig.tight_layout()
plt.show()


## Source and limitation

Source: [GEUS Report 2022/29](https://data.geus.dk/pure-pdf/GEUS-R_2022-29_web.pdf), input Tables 8.1.5.1–8.1.5.3 (report page 44), tornado Figure 8.2.2 for the Gassum Formation, and results Tables 8.2.1–8.2.4 (page 46).

The separate Haldager Sand and Skagerrak tornadoes and the combined Inez tornado are notebook extensions; they are not figures reproduced from the GEUS report.

This is static volumetric screening capacity. It does not yet represent pressure constraints, injectivity, plume migration, dynamic reservoir simulation or economics.
